# imports

In [ ]:
import torchimport torchvisionfrom torch import nnimport torchvision.transforms as transformsfrom tqdm.auto import tqdmimport warningsfrom timeit import default_timer as timerimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import confusion_matrixwarnings.filterwarnings('ignore')

## using CUDA

In [192]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Timer

In [193]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## Transforming to Tensors

In [194]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

In [195]:
trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

## 'Train', 'Validation', 'Test'

In [196]:
train_size = int(0.8 * len(trainset))val_size = len(trainset) - train_sizetrain_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size])# Create DataLoaders for each settrainloader = torch.utils.data.DataLoader(train_subset, batch_size=64, shuffle=True)valloader = torch.utils.data.DataLoader(val_subset, batch_size=64, shuffle=False)testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

## Deep Neural Network

In [197]:
class DeepNN(nn.Module):    def __init__(self, input_features=784, output_features=10, hidden_features=104, init_method=None):        super().__init__()        self.layer_stack = nn.Sequential(            nn.Linear(in_features=input_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=output_features),        )        self._initialize_weights(init_method)    def _initialize_weights(self, init_method):        if init_method != None:            for layer in self.layer_stack:                if isinstance(layer, nn.Linear):                    if init_method == 'he':                        nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')                    elif init_method == 'xavier':                        nn.init.xavier_normal_(layer.weight)                    elif init_method == 'random':                        nn.init.normal_(layer.weight, mean=0, std=1)                    nn.init.zeros_(layer.bias)  # Initialize biases to zero    def forward(self, x):        return self.layer_stack(x)